# Semana 07: Primeiro Acesso ao AWS Learner Lab Sandbox — Deploy da Fábrica Virtual Smart N1 com Docker Compose na EC2

## Integração Prática Nuvem AWS & Automação Industrial 4.0

Bem-vindos ao ambiente real de nuvem da **Amazon Web Services (AWS)**! A partir desta semana, conectamos a infraestrutura em nuvem com as aplicações de **Automação Industrial**.

Em vez de um simples servidor de teste genérico, nesta aula iremos provisionar uma instância **Amazon EC2 (Amazon Linux 2023)** no **AWS Academy Learner Lab** e realizar o deploy da stack completa da **Fábrica Virtual Smart N1** utilizando **Docker Compose**:
1. **Simulador SCADA da Fábrica Virtual (Flask / Python)** na porta `5000`.
2. **Node-RED (Orquestração de Fluxos IIoT)** na porta `1880`.
3. **Eclipse Mosquitto (Broker MQTT)** na porta `1883`.

Tudo isso descompactando o pacote oficial da disciplina (`semana_05.zip`) e subindo a infraestrutura com 1 único comando `docker compose`!

### Objetivos da Aula
- Operar o ambiente **AWS Academy Learner Lab Sandbox** e gerenciar sua cota de US$ 100 de créditos.
- Configurar um **Security Group** liberando as portas industriais (22, 5000, 1880 e 1883).
- Provisionar uma máquina virtual **Amazon EC2** com a AMI oficial **Amazon Linux 2023**.
- Conectar à instância via **EC2 Instance Connect** com o usuário padrão `ec2-user`.
- Instalar o **Docker Engine**, o utilitário **unzip** e o **Docker Compose** via repositórios nativos e oficiais.
- Baixar o arquivo `semana_05.zip`, descompactar e orquestrar os contêineres.
- Acessar o painel SCADA e o Node-RED externamente através do **IP Público** da EC2.
- Executar o procedimento de **Stop Instance** e **End Lab** para economia rigorosa de créditos.

---


## 1. Arquitetura da Solução na AWS

```text
+---------------------------------------------------------------------------------------------------------+
| AWS Cloud (Região us-east-1 - N. Virginia)                                                              |
|                                                                                                         |
|   VPC Padrão (Default VPC)                                                                              |
|   +-------------------------------------------------------------------------------------------------+   |
|   | Security Group (sg-smartn1-automacao)                                                           |   |
|   |   -> Porta 22   (SSH)  : Acesso administrativo via EC2 Instance Connect                         |   |
|   |   -> Porta 5000 (HTTP) : Painel SCADA / Simulador da Fábrica Virtual (Flask)                   |   |
|   |   -> Porta 1880 (HTTP) : Painel de Fluxos IIoT do Node-RED                                      |   |
|   |   -> Porta 1883 (TCP)  : Broker MQTT Mosquitto (opcional para sensores externos)                |   |
|   |                                                                                                 |   |
|   |   +-----------------------------------------------------------------------------------------+   |   |
|   |   | Instância Amazon EC2 (Amazon Linux 2023 - t2.micro / t3.micro)                          |   |   |
|   |   | Usuário: ec2-user  |  IP Público IPv4: ex. 54.210.xx.xx                                  |   |   |
|   |   |                                                                                         |   |   |
|   |   |   +---------------------------------------------------------------------------------+   |   |   |
|   |   |   | REDE DOCKER (rede_automacao via docker-compose)                                 |   |   |   |
|   |   |   |                                                                                 |   |   |   |
|   |   |   |   [1] Container: flask_simulator_app (Porta 5000:5000)                          |   |   |   |
|   |   |   |       -> Painel SCADA web em tempo real (Temperatura, Vibração, E-Stop)         |   |   |   |
|   |   |   |                               |                                                 |   |   |   |
|   |   |   |                               | Publica telemetria via MQTT (:1883)             |   |   |   |
|   |   |   |                               v                                                 |   |   |   |
|   |   |   |   [2] Container: mosquitto_broker (Porta 1883:1883 / 9001:9001)                 |   |   |   |
|   |   |   |       -> Roteamento de tópicos fabris (fabrica/#)                               |   |   |   |
|   |   |   |                               ^                                                 |   |   |   |
|   |   |   |                               | Subscreve tópicos de telemetria e alarmes       |   |   |   |
|   |   |   |                               |                                                 |   |   |   |
|   |   |   |   [3] Container: nodered_app (Porta 1880:1880)                                  |   |   |   |
|   |   |   |       -> Orquestração lógica de regras e automação industrial                   |   |   |   |
|   |   |   |                                                                                 |   |   |   |
|   |   |   +---------------------------------------------------------------------------------+   |   |   |
|   |   +-----------------------------------------------------------------------------------------+   |   |
|   +-------------------------------------------------------------------------------------------------+   |
+---------------------------------------------------------------------------------------------------------+
                                      ^
                                      | Acesso Web (Portas 5000 e 1880)
                                      |
                              [ Aluno / Navegador ]
```

---


## 2. O Ambiente Sandbox: AWS Academy Learner Lab

O **Learner Lab** disponibiliza uma conta real da AWS com **US$ 100 de orçamento pré-pago** para cada estudante.

| Característica | Detalhe operacional |
| :--- | :--- |
| **Orçamento** | US$ 100,00 por aluno para o semestre inteiro. |
| **Duração da Sessão** | Máximo de 4 horas contínuas. Pode ser renovada/reiniciada com *Start Lab*. |
| **Região Obrigatória** | **`us-east-1` (N. Virginia)**. Todos os recursos devem ser criados nesta região. |
| **Persistência de Dados** | Ao clicar em *End Lab*, as instâncias sofrem **Stop**, mas os discos EBS, arquivos e containers **NÃO são deletados**. |
| **Credenciais de Acesso** | Chave SSH padrão **`vockey`** pré-configurada no laboratório. |
| **Permissões IAM** | Utilize sempre a **`LabRole`** ou o **`LabInstanceProfile`**. |

> **REGRA DE OURO — ECONOMIA DE CRÉDITOS:**
> Ao encerrar o laboratório, **SEMPRE** pare suas instâncias EC2 (`Stop Instance`) e clique em **End Lab**. Nunca deixe instâncias ligadas sem uso.

---


## 3. Roteiro Prático Passo a Passo

### Etapa 1: Iniciar o Laboratório no AWS Academy

1. Acesse o portal da instituição no **AWS Academy** (Canvas LMS).
2. Abra o curso **AWS Academy Learner Lab**.
3. No menu, clique em **Learner Lab** para abrir o painel da sandbox.
4. No canto superior direito, clique no botão **Start Lab**:
   - Aguarde de 1 a 3 minutos até que o círculo fique **verde** com a indicação **ready**.
5. Quando estiver verde, clique no link com texto **AWS** (ao lado do círculo verde):
   - Uma nova aba do navegador abrirá diretamente no **Console de Gerenciamento da AWS**.
6. **Atenção:** Confira no canto superior direito do Console AWS se a região selecionada é **N. Virginia (`us-east-1`)**.

---


### Etapa 2: Criar o Security Group da Automação Industrial

1. No Console AWS, pesquise por **EC2** na barra superior.
2. No menu lateral esquerdo, em **Network & Security**, clique em **Security Groups**.
3. Clique no botão laranja **Create security group**.
4. Preencha as informações básicas:
   - **Security group name:** `sg-smartn1-automacao`
   - **Description:** `Permite SSH e portas 5000, 1880 e 1883 para a Smart N1`
   - **VPC:** Mantenha a VPC padrão selecionada (*Default VPC*).
5. Na seção **Inbound rules** (Regras de Entrada), adicione as regras abaixo clicando em **Add rule**:

| Tipo (*Type*) | Protocolo | Intervalo de Portas (*Port range*) | Origem (*Source*) | Finalidade |
| :--- | :--- | :--- | :--- | :--- |
| **SSH** | TCP | `22` | **Anywhere-IPv4 (`0.0.0.0/0`)** | Conexão de terminal via EC2 Instance Connect |
| **Custom TCP** | TCP | `5000` | **Anywhere-IPv4 (`0.0.0.0/0`)** | Acesso ao Painel SCADA da Fábrica Virtual (Flask) |
| **Custom TCP** | TCP | `1880` | **Anywhere-IPv4 (`0.0.0.0/0`)** | Acesso ao Painel de Fluxos do Node-RED |
| **Custom TCP** | TCP | `1883` | **Anywhere-IPv4 (`0.0.0.0/0`)** | Broker MQTT Mosquitto (opcional/sensores) |

6. Role até o final e clique no botão laranja **Create security group**.

---


### Etapa 3: Lançar a Instância Amazon EC2 com Amazon Linux 2023

1. No menu lateral esquerdo do painel EC2, clique em **Instances** e depois em **Launch instances**.
2. Configure os parâmetros da instância:
   - **Name and tags:** `ec2-smartn1-automacao-lab`
   - **Application and OS Images (AMI):**
     - Selecione **Amazon Linux** (primeira aba padrão).
     - Verifique se está selecionada a **Amazon Linux 2023 AMI** (64-bit x86, Free tier eligible).
   - **Instance type:**
     - Selecione `t3.micro` ou `t2.micro`.
   - **Key pair (login):**
     - Selecione o par de chaves **`vockey`** (chave padrão do Learner Lab).
   - **Network settings (Configurações de Rede):**
     - Clique no botão **Edit** (à direita).
     - Garanta que **Auto-assign public IP** esteja como **Enable** (Habilitado).
     - Em **Firewall (security groups)**, selecione **Select existing security group**.
     - Marque a caixa do grupo: `sg-smartn1-automacao`.
   - **Configure storage:**
     - Mantenha o padrão: `8 GiB gp3` (Root volume).
   - **Advanced details (Detalhes avançados):**
     - Em **IAM instance profile**, selecione **`LabInstanceProfile`** (ou `LabRole`).
3. Clique no botão laranja **Launch instance** no painel resumo à direita.
4. Clique em **View all instances** e aguarde até que o **Instance state** passe para **Running**.

---


### Etapa 4: Conectar à Instância via EC2 Instance Connect

1. Na lista de instâncias EC2, selecione a caixa ao lado de `ec2-smartn1-automacao-lab`.
2. Anote e copie o **Public IPv4 address** (ex: `54.210.xx.xx`).
3. Clique no botão superior **Connect**.
4. Na aba **EC2 Instance Connect**, confirme que o campo *User name* está preenchido como **`ec2-user`**.
5. Clique em **Connect** (botão laranja inferior).
6. O terminal do navegador abrirá conectado diretamente no servidor na nuvem.

---


### Etapa 5: Instalar Docker, Unzip e Docker Compose no Amazon Linux 2023

No terminal da EC2, execute os comandos abaixo para preparar o ambiente:

```bash
# 1. Atualizar o sistema e instalar o Docker e o utilitário Unzip nativamente
sudo dnf update -y
sudo dnf install -y docker unzip

# 2. Iniciar o serviço do Docker e habilitar no boot
sudo systemctl enable --now docker

# 3. Adicionar o usuário ec2-user ao grupo docker (para rodar comandos sem sudo)
sudo usermod -aG docker ec2-user
```

Agora vamos instalar o **Docker Compose CLI Plugin** oficial:

```bash
# 4. Criar diretório de plugins e baixar o binário oficial do Docker Compose
sudo mkdir -p /usr/local/lib/docker/cli-plugins
sudo curl -SL https://github.com/docker/compose/releases/latest/download/docker-compose-linux-x86_64 -o /usr/local/lib/docker/cli-plugins/docker-compose
sudo chmod +x /usr/local/lib/docker/cli-plugins/docker-compose

# 5. Atualizar a sessão do grupo docker
newgrp docker
```

Valide se ambos estão operacionais:

```bash
docker version
docker compose version
```

---


### Etapa 6: Baixar o Arquivo semana_05.zip e Descompactar

Agora vamos baixar o arquivo compactado `semana_05.zip` contendo toda a stack de Automação Industrial:

```bash
# Baixar o pacote semana_05.zip direto do repositório oficial da disciplina
curl -fsSL "https://raw.githubusercontent.com/profAndreSouza/Material/main/Automa%C3%A7%C3%A3o%20Industrial/materiais/semana_05.zip" -o semana_05.zip

# Conferir o tamanho do arquivo baixado
ls -lh semana_05.zip

# Descompactar o arquivo
unzip semana_05.zip

# Entrar na pasta extraída
cd semana_05

# Listar os arquivos do laboratório
ls -la
```

Você verá a pasta organizada com `docker-compose.yml`, `api/`, `mqtt/` e `nodered/`!

---


### Etapa 7: Subir a Stack Completa com Docker Compose

Com 1 único comando, o Docker Compose irá construir a imagem da API Flask, baixar o Mosquitto e o Node-RED e interconectar os 3 na rede virtual interna:

```bash
# Construir as imagens e iniciar os 3 contêineres em segundo plano
docker compose up -d --build
```

Acompanhe o status de inicialização:

```bash
# Listar os contêineres e suas respectivas portas
docker compose ps
```

A saída esperada deve mostrar os 3 serviços com status **Up**:
- `flask_simulator_app` ouvindo em `0.0.0.0:5000->5000/tcp`
- `nodered_app` ouvindo em `0.0.0.0:1880->1880/tcp`
- `mosquitto_broker` ouvindo em `0.0.0.0:1883->1883/tcp, 0.0.0.0:9001->9001/tcp`

Para inspecionar os logs em tempo real do simulador fabril:
```bash
docker compose logs -f flask_app
# (Pressione Ctrl+C para sair dos logs)
```

---


### Etapa 8: Acessar a Fábrica Virtual e o Node-RED no Navegador

Com a aplicação rodando na nuvem, abra o seu navegador e acesse:

#### 1. Painel SCADA da Fábrica Virtual (Flask)
```text
http://SEU_IP_PUBLICO:5000
```
- Você verá o painel de controle da **Smart N1** gerando telemetria em tempo real (Temperatura, Vibração, Pressão, Corrente).
- Teste os botões de simulação de anomalia: clique em **Superaquecimento**, **Vibração Excessiva** ou **Parada de Emergência (E-STOP)**.

#### 2. Painel de Fluxos IIoT (Node-RED)
```text
http://SEU_IP_PUBLICO:1880
```
- O ambiente de orquestração visual do Node-RED abrirá na nuvem.
- No menu superior direito `☰` > **Import**, cole o fluxo do arquivo `nodered/flows_semana05.json` e clique em **Deploy**.
- Abra a aba **Debug (ícone do inseto 🪲)** para visualizar os dados de telemetria e alarmes chegando via MQTT!

> **Dica de Diagnóstico:** Caso a página não abra no navegador:
> 1. Lembre-se de utilizar `http://` e nunca `https://`.
> 2. Certifique-se de que incluiu os dois pontos e a porta: `:5000` ou `:1880`.
> 3. Confira se o **Security Group** possui as regras de entrada liberando as portas 5000 e 1880 para `0.0.0.0/0`.

---


### Etapa 9: Procedimento Obrigatório de Final de Aula (Desligamento e Economia)

Para manter seu saldo de US$ 100 protegido para as próximas semanas:

1. No terminal da EC2, você pode parar os contêineres:
   ```bash
   docker compose stop
   ```
2. No Console da AWS, selecione a instância `ec2-smartn1-automacao-lab`.
3. Clique em **Instance state → Stop instance** (Parar instância).
   - **ATENÇÃO:** Nunca clique em *Terminate instance* (isso destruiria o disco e o laboratório). Apenas selecione **Stop instance**.
4. Aguarde o status mudar para **Stopped**.
5. Volte à aba do **AWS Academy Learner Lab** e clique no botão vermelho **End Lab**.

Na próxima aula, basta dar **Start Lab**, aguardar o círculo verde, ligar a EC2 (*Start instance*) e rodar `docker compose start` na pasta `semana_05`. Tudo estará pronto!

---


## 4. Exercícios de Avaliação e Entregáveis

Para comprovação da prática, envie as 3 evidências abaixo:

### Evidência 1: Print do Terminal Docker Compose
- Captura de tela do terminal da EC2 exibindo o resultado do comando:
  ```bash
  docker compose ps
  ```
  mostrando os 3 contêineres (`flask_simulator_app`, `nodered_app`, `mosquitto_broker`) em estado **Up** com suas portas mapeadas.

### Evidência 2: Print do Painel SCADA da Fábrica Virtual na Nuvem
- Captura de tela do navegador mostrando o painel da Fábrica Virtual acessado via `http://<IP_PUBLICO_EC2>:5000`, com gráficos de telemetria ativos.

### Evidência 3: Print do Node-RED na Nuvem
- Captura de tela do navegador em `http://<IP_PUBLICO_EC2>:1880` exibindo o fluxo importado e as mensagens no painel lateral de Debug (`🪲`).

### Questão Conceitual
Explique como os contêineres `flask_simulator_app` e `nodered_app` conseguem se comunicar com o `mosquitto_broker` utilizando apenas o nome do serviço (ex: `MQTT_BROKER_HOST=mosquitto`) sem precisar saber o endereço IP interno da máquina.